# AoC 2024 Day 7 — Bridge Repair

**Spark — aggregate fold over reachable partials**

Puzzle: <https://adventofcode.com/2024/day/7>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

> **Part 1 only.** Advent of Code reveals Part Two only after a correct Part One submission, and this day is unsolved — so part 2's text does not exist to work from yet. Submitting the answer below unlocks it.

---

## The puzzle

Each line is an equation with its operators stolen: a **test value**, a colon, then a list of operands.

```
3267: 81 40 27
```

Slot `+` or `*` into each gap, in any combination. Evaluation is strictly **left to right** — no operator precedence — and the operands cannot be reordered. A line is *solvable* if at least one choice of operators produces the test value; `81 + 40 * 27` and `81 * 40 + 27` both give 3267, so that line counts once.

- **Part 1** — sum the test values of the solvable lines.

## The approach

The obvious reading is "try every operator string": *n* operands means 2^(n−1) combinations, and the longest line here has **12 operands** — 2048 expressions to evaluate, per line, for 850 lines.

The reframing is to stop enumerating *operator strings* and start carrying **the set of values reachable so far**. After consuming *k* operands there are at most 2^(k−1) distinct partial results, and usually far fewer — that set is all the state the rest of the line needs.

Which is a fold, and Spark has one that runs on array columns: `aggregate`. Seed with `array(nums[0])`, and at each operand map every partial *v* to `{v+x, v*x}`, `flatten`, `array_distinct`. No UDF, no `collect` — the whole line is one native expression tree, and the driver only ever sees the final `sum`.

Two things keep the accumulator small:

- **Pruning.** Every operand in this input is ≥ 1, so both `+` and `*` are monotonically non-decreasing — a partial that has already passed the target can never come back down. `filter(v <= target)` drops it immediately, which is what stops the 12-operand lines from ever materialising 2048 partials.
- **Dedup.** `array_distinct` collapses branches that converge. 356 of the operands are `1`, where `v*1 == v` merges straight back into the `+` branch, and cases like `81+40*27` vs `81*40+27` both landing on 3267 collapse too.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day07

spark = get_spark('aoc-2024-day07')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = '190: 10 19\n3267: 81 40 27\n83: 17 5\n156: 15 6\n7290: 6 8 6 15\n161011: 16 10 13\n192: 17 8 14\n21037: 9 7 18 13\n292: 11 6 16 20\n'

print('part 1:', day07.part1(spark, EXAMPLE), '(expected 3749)')

### The fold, one operand at a time

The first table below is the fold *without* the pruning filter, so you can watch the reachable set double (and then not quite double, as duplicates collapse). The second is what the solution actually evaluates — same fold, clipped at the target on every step.

In [ ]:
from pyspark.sql import functions as F

eq = day07.parse(spark, EXAMPLE)
eq.show(truncate=False)


def reachable_after(k):
    """Partials after folding in the first k operands -- no pruning."""
    return F.aggregate(
        F.slice(F.col('nums'), 2, k - 1),
        F.array(F.col('nums').getItem(0)),
        lambda acc, x: F.array_distinct(
            F.flatten(F.transform(acc, lambda v: F.array(v + x, v * x)))
        ),
    )


# One line, one operand at a time: watch the reachable set fan out.
eq.filter(F.size('nums') == 4).select(
    'target',
    'nums',
    *[reachable_after(k).alias(f'after_{k}') for k in range(1, 5)],
).show(truncate=False)

# Now the real thing: pruned at every step, then tested for the target.
ops = [lambda v, x: v + x, lambda v, x: v * x]
reachable = day07._reachable(ops)
eq.select(
    'target',
    'nums',
    reachable.alias('reachable'),
    F.size(reachable).alias('n'),
    F.array_contains(reachable, F.col('target')).alias('solvable'),
).show(truncate=False)

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 7)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

started = time.perf_counter()
answer = day07.part1(spark, data)
print(f'part 1: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Cross-check

Days 6+ have no known-good answer, so correctness rests on an independently written plain-Python implementation agreeing with the Spark one. That is evidence, not proof — a shared misreading of the puzzle would survive both.

In [ ]:
from reference_python.y2024 import day07 as reference

cross = reference.part1(data)
print('reference:', cross)
print('agree:    ', cross == answer)

## Notes & gotchas

- `slice` is **1-based**, so `slice(nums, 2, size - 1)` means "every operand after the first". The first operand is the fold *seed*, not an element of the fold.
- The `v <= target` prune is only correct because **every operand is ≥ 1**. A single `0` would make `v * 0` collapse a too-large partial back into range, and a negative operand would do the same for `+`. This input has neither (min operand is 1) — check before reusing the trick.
- Targets run past 2^31 (the real answer is a 13-digit number), so `parse` casts to `bigint` on both sides. Under ANSI mode an `int` overflow **raises** rather than wrapping, so a missed cast fails loudly instead of quietly.
- `split(line, ': ')` takes a **regex**, not a literal. `': '` happens to contain no metacharacters; a separator like `|` or `.` would silently match everything.
- `_reachable` takes the operator list as an argument rather than hardcoding `+` and `*`. Adding an operator is a one-line change at the call site, not a rewrite of the fold.